In [20]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n

# compare against generated code
encoder_ouput = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_ouput = cw if encoder_ouput is None else np.vstack((encoder_ouput, cw))



Field Closed Succesfully!, 1 Non-Zero Elements


In [22]:
with open("golay_code_constraint.svh", "w") as file:
    file.write("\tconstraint golay_code {\n")
    file.write("\t\trx_data inside {\n")
    for cw in encoder_ouput:
        v = gf.do_pack(cw)

        if np.all(cw == encoder_ouput[-1]):
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")
    file.write("\t};\n")

## golay 24,12 decoder model

In [23]:
r = encoder_ouput[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_ouput[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [24]:
# decode all codewords, no errors
for cw in encoder_ouput:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_ouput:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


#### Test `golay_err_gen.sv`

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

Obtain values corrected.

In [25]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rs_words = [format(x, '024b') for x in rs]

def str_values_2_binary(r_n):

    # Convert text to bits.
    r_bits = np.array([int(bit) for bit in r_n])

    return r_bits

def array_2_binaryInt(w : np.array):
    # Convert str decode word into hexadecimal
    # 1. numbers into string map(str, w)
    # 2. put together .join()
    # 3. interprete as binary int(str, 2) 2 binary
    w_hex = int(''.join(map(str, w)), 2)
    return w_hex

with open("outputs/decoding/golay_decoding_test_vectors.txt", "w") as f:

    # for within rs_words
    # i: number of iterations
    # r_n: current word of rs_words
    for i, r_n in enumerate(rs_words, start=1):
        r_bits = str_values_2_binary(r_n)

        # Call method .decode
        w, corrected, uncorrectable = decoder_model.decode(
            r_bits,
            full_codeword=True
        )

        w_hex = array_2_binaryInt(w)   # Obtain HEX value from rs array
        r_hex = rs[i-1]

        f.write(f"{r_hex:06X} {w_hex:06X}\n")

        print(
            f"r{i} = 0x{r_hex:06X} | "
            f"corrected_w = 0x{w_hex:06X} | "
            f"corrected = {corrected} | "
            f"uncorrectable = {uncorrectable}"
        )

print("Archivo generado: golay_decoding_test_vectors.txt")

r1 = 0xA5D9A6 | corrected_w = 0xA5C9A5 | corrected = True | uncorrectable = False
r2 = 0xA5F9A4 | corrected_w = 0xA5C9A5 | corrected = True | uncorrectable = False
r3 = 0xA5C9AA | corrected_w = 0xA5C9AA | corrected = False | uncorrectable = True
Archivo generado: golay_decoding_test_vectors.txt


##### Obtain error



In [26]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rs_words = [format(x, '024b') for x in rs]

with open("outputs/error_gen/golay_err_gen_vector.txt", "w") as f:

    for i, r_n in enumerate(rs_words, start=1):

        r_bits = str_values_2_binary(r_n)
        s, q = decoder_model.get_s_q(r_bits)
        error = decoder_model.get_error(s, q)
        r_hex = rs[i - 1]

        ### Gets
        # sbi, qbi 
        for i, bi in enumerate(decoder_model._G2412B):
            sbi, qbi = decoder_model.get_sbi_qbi(s, q, bi)
            ui    = decoder_model._gf.do_unpack(1<<i, decoder_model._k)[::-1]

        w_syn = decoder_model._gf.hamming_weight(s)
        w_q   = decoder_model._gf.hamming_weight(q)

        _, _, uncorrectable = decoder_model.decode(r_bits)

        # prints
        if error is None:
            print(
                f"r{i} = 0x{r_hex:06X} | "
                f"error_mask = UNCORRECTABLE"
            )
            f.write(f"{r_hex:06X} UNCORRECTABLE\n")
        else:
            hex_error_mask = array_2_binaryInt(error)
            f.write(
                f"{r_hex:06X} "
                f"{hex_error_mask:06X}\n"
            )

            print(
                f"r{i} = 0x{r_hex:06X} | "
                f"error_mask_{i} = "
                f"0x{hex_error_mask:06X}"
            )

r11 = 0xA5D9A6 | error_mask_11 = 0x001003
r11 = 0xA5F9A4 | error_mask_11 = 0x003001
r11 = 0xA5C9AA | error_mask = UNCORRECTABLE


In [ ]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]
rs_words = [format(x, "024b") for x in rs]

with open("outputs/error_gen/golay_err_gen_vector.txt", "w") as f:

    for idx, r_n in enumerate(rs_words, start=1):

        r_bits = str_values_2_binary(r_n)
        r_hex = rs[idx - 1]

        # Syndrome, q and error
        s, q = decoder_model.get_s_q(r_bits)
        error = decoder_model.get_error(s, q)

        # Find s and q indices independently
        s_idx = None
        q_idx = None
        sbi   = None
        qbi   = None
        s_ui  = None
        q_ui  = None

        #decoder_model._G2412B



r1 = 0xA5D9A6
----------------------------------------------
s             = [1 1 1 0 1 0 1 0 1 0 1 0] | 0xEAA
q             = [0 0 0 1 1 0 1 1 0 0 1 0] | 0x1B2
----------------------------------------------
s_index       = 11
sbi           = [0 0 0 0 0 0 0 0 0 0 1 1] | 0x003
s_ui          = [0 0 0 0 0 0 0 0 0 0 0 1] | 0x001
----------------------------------------------
q_index       = None
qbi           = None | 0xNone
q_ui          = None | 0xNone
----------------------------------------------
w_syn         = 7
w_q           = 5
uncorrectable = False
----------------------------------------------
error_mask    = 001003

r2 = 0xA5F9A4
----------------------------------------------
s             = [0 0 0 1 1 0 1 1 0 0 1 0] | 0x1B2
q             = [1 1 1 0 1 0 1 0 1 0 1 0] | 0xEAA
----------------------------------------------
s_index       = None
sbi           = None | 0xNone
s_ui          = None | 0xNone
----------------------------------------------
q_index       = 11
qbi          